In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, ShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from scipy.stats import zscore

In [23]:
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    root_mean_squared_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr
import os

def metrics(method, y_test, y_pred):
    """
    Outputs classification metrics
    """
    #config_path = os.path.join(os.path.dirname(__file__), "../config.yaml")
    #with open(config_path) as f:
    #    config = yaml.safe_load(f)
    
    corr, p_value = pearsonr(y_test, y_pred)
    print(f'{method}. Classification metrics:\n')
    print(f"MAE: {mean_absolute_error(y_test, y_pred)}")
    print(f"RMSE: {root_mean_squared_error(y_test, y_pred)}")
    print(f"R²: {r2_score(y_test, y_pred)}")
    print(f"Pearson correlation coefficient: {corr}, p-value: {p_value}\n")
    
    return r2_score(y_test, y_pred)
    

In [4]:
def split_data(descriptors_path, features_file, method='RandomForestRegressor', test_size=0.1, random_state=18):

    data = pd.read_csv(descriptors_path)
    data = data.dropna()
    
    features = pd.read_csv(features_file)
    RF_features = features['RF']
    DT_features = features['DT']
    GB_features = features['GB']

    feature_sets = {"RandomForestRegressor": RF_features, 
                    "DecisionTreeRegressor": DT_features,
                   "GradientBoostingRegressor": GB_features}
    
    # Selection of specified features and target variable
    features = feature_sets[method]
    X = data[features]  # Selected features
    
    z_scores = zscore(X)
    X_filtered = X[(z_scores < 3).all(axis=1)]
    y_filtered = data.loc[X_filtered.index, "pIC50"].str.replace(',', '.').astype(float)
    
    # Нормализация данных
    categorial_col = [col for col in X_filtered.columns if X_filtered[col].nunique() == 2 and int(X_filtered[col].max())==1]
    numeric_col = [col for col in X_filtered.columns if not(X_filtered[col].nunique() == 2 and int(X_filtered[col].max())==1)]
    
    # categorial_col = [col for col in X.columns if X[col].nunique() == 2 and int(X[col].max())==1]
    # numeric_col = [col for col in X.columns if not(X[col].nunique() == 2 and int(X[col].max())==1)]
    preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_col),
        ('cat', 'passthrough', categorial_col) 
    ])

    X_scaled = preprocessor.fit_transform(X)

    # Преобразуем результат обратно в DataFrame (опционально)
    X_scaled = pd.DataFrame(X_scaled, columns=numeric_col + categorial_col)
    # scaler = StandardScaler()
    # X = scaler.fit_transform(X)
    y = data["pIC50"]  # Target value
    y = y.str.replace(',', '.').astype(float)
    
    # Разделение данных на обучающую и тестовую выборки
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=test_size, random_state=random_state)
    
    return X_train, X_test, y_train, y_test

In [7]:
X_train, X_test, y_train, y_test = split_data('NS3_4A_descriptors.csv', 'my_features_NS3_4A.csv', method='GradientBoostingRegressor')

In [8]:
from sklearn.svm import SVR

def support_vector_regression(X_train, X_test, y_train, y_test):
    
    svr = SVR()

    # Parameters for selection
    param_grid = {
        'kernel': ['rbf', 'poly'],
        'C': [0.1, 1],  # Regularization parameter
        'gamma': [0.0001, 0.001],  # Core coefficient
        'epsilon': [0.01, 0.1]  
    }
    
    cv = ShuffleSplit(n_splits=10, test_size=0.2, random_state=18)
    # Cross-validation with GridSearchCV
    grid_search = GridSearchCV(
        estimator=svr,
        param_grid=param_grid,
        scoring='neg_mean_squared_error',  # Optimization metric
        cv=cv,  # Number of folds
        n_jobs=-1,  # Use all CPU cores
    )

    # Training
    grid_search.fit(X_train, y_train)

    # Best parametrs
    print(f"Best parametrs: {grid_search.best_params_} \n")
    print('_______________________________________________________________')

    # Prediction
    y_pred = grid_search.predict(X_test)

    # Model evaluation
    print("Test metrics")
    test_metric = metrics("Support vector regression", y_test, y_pred)
    
    print('_______________________________________________________________')
    print("Train metrics")
    train_metric = metrics("Support vector regression", y_train, grid_search.predict(X_train))
    
    return test_metric, train_metric
    


In [24]:
svr_test_metric, svr_train_metric = support_vector_regression(X_train, X_test, y_train, y_test)

Best parametrs: {'C': 1, 'epsilon': 0.01, 'gamma': 0.001, 'kernel': 'rbf'} 

_______________________________________________________________
Test metrics
Support vector regression. Classification metrics:

MAE: 0.6893711988501832
RMSE: 0.9194249130729426
R²: 0.6883023074772411
Pearson correlation coefficient: 0.8673670455619665, p-value: 7.563319566196865e-16

_______________________________________________________________
Train metrics
Support vector regression. Classification metrics:

MAE: 0.599574186442417
RMSE: 0.8082586631921319
R²: 0.6959370483193572
Pearson correlation coefficient: 0.8422075205974231, p-value: 4.8166029358701095e-119



In [12]:
from sklearn.neighbors import KNeighborsRegressor

def k_Nearest_Neighbour(X_train, X_test, y_train, y_test):
    
    knn = KNeighborsRegressor()
    
    # Parameters for selection
    param_grid = {
    'n_neighbors': [7, 9, 11]
    }
    
    # Cross-validation with GridSearchCV
    grid_search = GridSearchCV(
        estimator=knn,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',  # Optimization metric
        cv=10,  # Number of folds
        n_jobs=-1,  # Use all CPU cores
    )

    # Training
    grid_search.fit(X_train, y_train)

    # Best parametrs
    print("Best parametrs:", grid_search.best_params_)
    print('_______________________________________________________________')

    # Prediction
    y_pred = grid_search.predict(X_test)

    # Model evaluation
    print("Test metrics")
    test_metric = metrics("k Nearest Neighbour", y_test, y_pred)
    
    print('_______________________________________________________________')
    print("Train metrics")
    train_metric = metrics("k Nearest Neighbour", y_train, grid_search.predict(X_train))
    
    return test_metric, train_metric
    

In [25]:
knn_test_metric, knn_train_metric = k_Nearest_Neighbour(X_train, X_test, y_train, y_test)

Best parametrs: {'n_neighbors': 7}
_______________________________________________________________
Test metrics
k Nearest Neighbour. Classification metrics:

MAE: 0.49980515816618076
RMSE: 0.6949155502103818
R²: 0.821940515144674
Pearson correlation coefficient: 0.9071378496829433, p-value: 2.7365874209452216e-19

_______________________________________________________________
Train metrics
k Nearest Neighbour. Classification metrics:

MAE: 0.4439384288003914
RMSE: 0.6082221912533912
R²: 0.8278181801645045
Pearson correlation coefficient: 0.9106667259995592, p-value: 1.74417559172115e-169



In [20]:
from sklearn.neural_network import MLPRegressor

def MLP_regressor(X_train, X_test, y_train, y_test):
    
    #mlp = MLPRegressor()
    mlp = MLPRegressor(early_stopping=True, validation_fraction=0.2, random_state=44)
    
    param_grid = {
    'hidden_layer_sizes': [(2,), (5,), (10,)],
    #'activation': ['logistic', 'tanh','relu'],
    #'solver': ['sgd', 'adam'],
    'learning_rate': ['adaptive'],
    'alpha': [0.0001, 0.001, 0.01, 0.1, 0.5],
    'learning_rate_init': [0.00001, 0.0001, 0.001],
    'max_iter': [50000]
}
    
    cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=18)
     # Cross-validation with GridSearchCV
    grid_search = GridSearchCV(
        estimator=mlp,
        param_grid=param_grid,
        scoring='neg_mean_squared_error',  # Optimization metric
        cv=cv,  # Number of folds
        n_jobs=-1,  # Use all CPU cores
    )

    # Training
    grid_search.fit(X_train, y_train)

    # Best parametrs
    print("Best parametrs:", grid_search.best_params_)
    print('_______________________________________________________________')

    # Prediction
    y_pred = grid_search.predict(X_test)

    # Model evaluation
    print("Test metrics")
    test_metric = metrics("Multi-layer Perceptron regressor", y_test, y_pred)
    
    print('_______________________________________________________________')
    print("Train metrics")
    train_metric = metrics("Multi-layer Perceptron regressor", y_train, grid_search.predict(X_train))
    
    return test_metric, train_metric

In [26]:
mlp_test_metric, mlp_train_metric = MLP_regressor(X_train, X_test, y_train, y_test)

/home/pgurzhii/miniforge3/envs/HCV/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pgurzhii/miniforge3/envs/HCV/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pgurzhii/miniforge3/envs/HCV/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pgurzhii/miniforge3/envs/HCV/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50000) reached and the optimization hasn't converged ye

Best parametrs: {'alpha': 0.5, 'hidden_layer_sizes': (5,), 'learning_rate': 'adaptive', 'learning_rate_init': 0.001, 'max_iter': 50000}
_______________________________________________________________
Test metrics
Multi-layer Perceptron regressor. Classification metrics:

MAE: 0.7459307038022349
RMSE: 0.9642587705882446
R²: 0.6571625564640851
Pearson correlation coefficient: 0.8507447286351275, p-value: 1.0010380679558478e-14

_______________________________________________________________
Train metrics
Multi-layer Perceptron regressor. Classification metrics:

MAE: 0.639052450497239
RMSE: 0.8216348593738052
R²: 0.6857896517305726
Pearson correlation coefficient: 0.8449749505255328, p-value: 1.4068264947667425e-120



In [27]:
metrics = pd.DataFrame()
metrics["GB_train"] = svr_train_metric, knn_train_metric, mlp_train_metric
metrics["GB_test"] = svr_test_metric, knn_test_metric, mlp_test_metric
metrics.index = ['SVR', 'kNN', 'MLP'] 

print(metrics)
metrics.to_csv("R2_GB.csv")

     GB_train   GB_test
SVR  0.695937  0.688302
kNN  0.827818  0.821941
MLP  0.685790  0.657163
